# MRIQC — Batch Job Submission

This notebook auto-detects all subjects in your BIDS directory, generates one Slurm job script per subject, and prints the `sbatch` commands to submit them to the HPC cluster in parallel.

**Before running this notebook:**
- Confirm MRIQC runs correctly on a single subject using `mriqc_single.bsh`
- Ensure all subjects have been BIDS-converted and are present in `bids_data/`

**Reference:** [MRIQC documentation](https://mriqc.readthedocs.io/en/stable/about.html)

## 1. Imports

In [ ]:
import os
import glob

## 2. Set Variables

Edit `project` — all other paths are derived from it automatically.

In [ ]:
# ── Project ───────────────────────────────────────────────────────────────────
project     = 'your_project'   # Project folder name under /data00/projects/
project_dir = os.path.join('/data00/projects/', project)

# ── Paths ─────────────────────────────────────────────────────────────────────
bids_dir    = os.path.join(project_dir, 'data/bids_data')
output_dir  = os.path.join(bids_dir, 'derivatives/mriqc')
working_dir = os.path.join(bids_dir, 'derivatives/working')
slurm_dir   = os.path.join(project_dir, 'scripts/BIDS/jobs/mriqc')
mriqc_sif   = '/data00/tools/singularity_images/mriqc-0.15.1.simg'

# ── Create output directories ─────────────────────────────────────────────────
os.makedirs(output_dir, exist_ok=True)
os.makedirs(working_dir, exist_ok=True)
os.makedirs(slurm_dir, exist_ok=True)
os.makedirs(os.path.join(slurm_dir, 'out'), exist_ok=True)  # for .out/.err logs

print(f"Project     : {project}")
print(f"BIDS dir    : {bids_dir}")
print(f"Output dir  : {output_dir}")
print(f"Slurm dir   : {slurm_dir}")

## 3. Detect Subjects

Subjects are auto-detected by finding all `sub-*` directories in the BIDS folder. The `sub-` prefix is stripped because MRIQC's `--participant_label` flag expects the bare ID.

In [ ]:
# Auto-detect all BIDS subject directories and strip the 'sub-' prefix
subs = sorted([
    os.path.basename(d).replace('sub-', '')
    for d in glob.glob(os.path.join(bids_dir, 'sub-*'))
    if os.path.isdir(d)
])

print(f"Found {len(subs)} subjects: {subs}")

## 4. Define Slurm Job Template

Each job runs MRIQC on a single participant via Singularity.

**Adjust if needed:**
- `--time` — default is 2 days; increase for datasets with many long BOLD runs
- `--nprocs` and `-c` — number of CPUs requested; both should match
- `-m bold` — modality filter; remove or change to `T1w T2w bold` to include structural scans

In [ ]:
job_template = r'''#!/bin/bash
#SBATCH --job-name=mriqc_{ID}
#SBATCH --output=out/mriqc_{ID}.out
#SBATCH --error=out/mriqc_{ID}.err
#SBATCH --time=2-00:00:00
#SBATCH --cpus-per-task=8

srun singularity run --cleanenv \
    -B {bids_dir}:/data \
    -B {output_dir}:/out \
    -B {working_dir}:/work \
    {mriqc_sif} /data /out participant \
    --nprocs 8 \
    -m bold \
    --work-dir /work \
    --participant_label {ID}
'''

## 5. Generate Job Scripts

Writes one `.job` file per subject into the Slurm jobs directory.

In [ ]:
for sub in subs:
    job_content = job_template.format(
        ID=sub,
        bids_dir=bids_dir,
        output_dir=output_dir,
        working_dir=working_dir,
        mriqc_sif=mriqc_sif
    )

    job_path = os.path.join(slurm_dir, f'mriqc_{sub}.job')
    with open(job_path, 'w') as f:
        f.write(job_content)

    print(f'Written: {job_path}')

print(f'\nAll {len(subs)} job scripts written to: {slurm_dir}')

## 6. Print sbatch Commands

Copy and paste this output into a terminal after SSH-ing to the Slurm master node:


In [ ]:
print(f'cd {slurm_dir}\n')
for sub in subs:
    print(f'sbatch -D {slurm_dir} -c 8 mriqc_{sub}.job')